# 📈 Stock Price Predictor — End-to-End ML Pipeline

**Intern:** Ammar Akbar | **Organization:** DevelopersHub Corporation

**Objective:** Predict next-day stock closing prices using historical data, technical indicators, and regression models.

---

## 1. Setup & Imports

We import all necessary libraries:
- `yfinance` — download stock data from Yahoo Finance
- `pandas` / `numpy` — data manipulation
- `sklearn` — machine learning models & metrics
- `plotly` — interactive charts
- `ta` — technical analysis indicators

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Stock data API
import yfinance as yf

# Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# Technical indicators
import ta

print('All libraries imported successfully!')

## 2. Data Loading

### Why yfinance?
yfinance is a free Python library that downloads historical stock data directly from Yahoo Finance. No API key needed!

### What data do we get?
- **Open** — price when market opened
- **High** — highest price of the day
- **Low** — lowest price of the day
- **Close** — price when market closed (our **target**)
- **Volume** — number of shares traded

In [ ]:
# ============================================
# STEP 2A: Download stock data using yfinance
# ============================================
# We'll use Apple (AAPL) as our stock
# Downloading 3 years of daily data

ticker = 'AAPL'
start_date = '2022-01-01'
end_date = '2025-01-01'

print(f'Downloading {ticker} data from {start_date} to {end_date}...')
df = yf.download(ticker, start=start_date, end=end_date, progress=False)

# Flatten multi-level columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'Downloaded {len(df)} rows of data')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')
df.head()

In [ ]:
# ============================================
# STEP 2B: Basic data exploration
# ============================================
print('=== Dataset Shape ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
print()
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Statistical Summary ===')
df.describe()

## 3. Data Preprocessing

Before we can train a model, we need to clean the data:
1. **Remove duplicates** — same date appearing twice
2. **Handle missing values** — fill forward (use yesterday's price)
3. **Sort by date** — ensure chronological order (critical for time series!)

In [ ]:
# ============================================
# STEP 3: Clean the data
# ============================================

# Remove duplicate dates
df = df[~df.index.duplicated(keep='first')]

# Sort by date (chronological order)
df = df.sort_index()

# Forward-fill missing values
# WHY: If a stock was $150 yesterday and data is missing today,
# the best guess is still $150 (not zero or mean)
df = df.ffill()

# Drop any remaining NaN rows
df = df.dropna()

print(f'After cleaning: {len(df)} rows')
print(f'Missing values: {df.isnull().sum().sum()}')

## 4. Exploratory Data Analysis (EDA)

Let's visualize the stock price history to understand patterns before building models.

In [ ]:
# ============================================
# STEP 4A: Interactive candlestick chart
# ============================================
# A candlestick chart shows Open, High, Low, Close for each day
# Green = price went UP, Red = price went DOWN

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    vertical_spacing=0.05, row_heights=[0.75, 0.25],
                    subplot_titles=(f'{ticker} Stock Price', 'Trading Volume'))

fig.add_trace(go.Candlestick(
    x=df.index, open=df['Open'], high=df['High'],
    low=df['Low'], close=df['Close'], name='OHLC',
    increasing_line_color='#22c55e', decreasing_line_color='#ef4444'
), row=1, col=1)

# Volume bars
colors = ['#22c55e' if c >= o else '#ef4444' for c, o in zip(df['Close'], df['Open'])]
fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume',
                     marker_color=colors, opacity=0.5), row=2, col=1)

fig.update_layout(xaxis_rangeslider_visible=False, template='plotly_dark',
                  height=600, showlegend=False, title=f'{ticker} — Price & Volume')
fig.show()

In [ ]:
# ============================================
# STEP 4B: Price distribution & statistics
# ============================================
print('=== Key Statistics ===')
print(f'Average Close: ${df["Close"].mean():.2f}')
print(f'Highest Close: ${df["Close"].max():.2f}')
print(f'Lowest Close:  ${df["Close"].min():.2f}')
print(f'Std Dev:       ${df["Close"].std():.2f}')
print(f'Total Return:  {((df["Close"].iloc[-1] - df["Close"].iloc[0]) / df["Close"].iloc[0] * 100):.1f}%')

## 5. Feature Engineering

Raw stock prices alone aren't very informative for prediction. We need to create **features** that capture patterns:

| Feature Type | What It Captures |
|---|---|
| **Lag features** | Yesterday's price, 2 days ago, etc. (autocorrelation) |
| **Moving Averages** | Short/long term trends (SMA_7, SMA_21, SMA_50) |
| **RSI** | Whether stock is overbought (>70) or oversold (<30) |
| **Daily Returns** | Percentage price change (momentum) |
| **Volume Ratio** | Whether trading activity is above/below average |

In [ ]:
# ============================================
# STEP 5: Create all features
# ============================================
df_feat = df.copy()

# ---- LAG FEATURES ----
# Yesterday's close is the STRONGEST predictor
# Because stock prices are highly autocorrelated
for lag in [1, 2, 3, 5, 7]:
    df_feat[f'Close_lag_{lag}'] = df_feat['Close'].shift(lag)

# ---- MOVING AVERAGES ----
# SMA_7  = short-term trend (1 week)
# SMA_21 = medium-term trend (1 month)
# SMA_50 = long-term trend (2 months)
for window in [7, 21, 50]:
    df_feat[f'SMA_{window}'] = df_feat['Close'].rolling(window=window).mean()

# ---- RSI (Relative Strength Index) ----
# Measures momentum: RSI > 70 = overbought, RSI < 30 = oversold
df_feat['RSI_14'] = ta.momentum.RSIIndicator(df_feat['Close'], window=14).rsi()

# ---- DAILY RETURNS ----
# Percentage change from yesterday
df_feat['daily_return'] = df_feat['Close'].pct_change() * 100

# ---- PRICE RANGE & OPEN-CLOSE DIFF ----
df_feat['price_range'] = df_feat['High'] - df_feat['Low']
df_feat['open_close_diff'] = df_feat['Close'] - df_feat['Open']

# ---- VOLUME FEATURES ----
df_feat['volume_change'] = df_feat['Volume'].pct_change() * 100
df_feat['volume_ratio'] = df_feat['Volume'] / df_feat['Volume'].rolling(20).mean()

# ---- TARGET VARIABLE ----
# We predict TOMORROW's close price
df_feat['target'] = df_feat['Close'].shift(-1)

# Drop rows with NaN (from lag/rolling calculations)
df_feat = df_feat.dropna()

print(f'Features created: {df_feat.shape[1] - 1} features + 1 target')
print(f'Dataset size: {len(df_feat)} rows')
print()
print('Feature columns:')
for col in df_feat.columns:
    if col != 'target':
        print(f'  • {col}')

## 6. Train/Test Split (Chronological)

### ⚠️ Critical: Why NOT random split?
In time series, future data must NEVER leak into training. We split **chronologically**:
- **Train**: First 80% of data (older dates)
- **Test**: Last 20% of data (recent dates)

Random splitting would let the model "peek" at future prices — that's **data leakage**!

In [ ]:
# ============================================
# STEP 6: Chronological train/test split
# ============================================

# Features = everything EXCEPT raw prices and target
# WHY exclude raw prices? They'd cause data leakage!
# The model would just copy Close to predict target
exclude_cols = ['Close', 'Open', 'High', 'Low', 'Adj Close', 'Volume', 'target']
feature_cols = [c for c in df_feat.columns if c not in exclude_cols]

X = df_feat[feature_cols]
y = df_feat['target']

# Split at 80% mark
split_idx = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Training set: {len(X_train)} samples ({X_train.index[0].date()} to {X_train.index[-1].date()})')
print(f'Test set:     {len(X_test)} samples ({X_test.index[0].date()} to {X_test.index[-1].date()})')
print(f'Features:     {len(feature_cols)}')
print()
print('Feature list:')
for f in feature_cols:
    print(f'  {f}')

## 7. Feature Scaling

We use **StandardScaler** to normalize features to have mean=0 and std=1.

### Why scale?
- Features have different ranges (RSI: 0-100, price: $100-$300, volume_ratio: 0-5)
- Models like Linear Regression are sensitive to feature magnitudes
- Scaling ensures all features contribute equally

### ⚠️ Important: Fit scaler ONLY on training data, then transform both train & test

In [ ]:
# ============================================
# STEP 7: Scale features
# ============================================
scaler = StandardScaler()

# Fit on training data ONLY (prevent data leakage)
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns, index=X_train.index
)

# Transform test data using the SAME scaler
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns, index=X_test.index
)

print('Scaling complete!')
print(f'Train mean (should be ~0): {X_train_scaled.mean().mean():.6f}')
print(f'Train std  (should be ~1): {X_train_scaled.std().mean():.6f}')

## 8. Model Training

We train two models:

| Model | Type | Strengths |
|---|---|---|
| **Linear Regression** | Parametric | Simple, interpretable, good with linear trends |
| **Random Forest** | Ensemble (trees) | Handles non-linearity, robust to outliers |

In [ ]:
# ============================================
# STEP 8A: Train Linear Regression
# ============================================
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)

# Evaluate
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

print('=== Linear Regression Results ===')
print(f'MAE:  ${lr_mae:.2f}  (avg prediction error)')
print(f'RMSE: ${lr_rmse:.2f}  (penalizes large errors)')
print(f'R²:   {lr_r2:.4f}   (1.0 = perfect, 0.0 = baseline)')

In [ ]:
# ============================================
# STEP 8B: Train Random Forest
# ============================================
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
rf_preds = rf_model.predict(X_test_scaled)

# Evaluate
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

print('=== Random Forest Results ===')
print(f'MAE:  ${rf_mae:.2f}')
print(f'RMSE: ${rf_rmse:.2f}')
print(f'R²:   {rf_r2:.4f}')

## 9. Model Comparison

In [ ]:
# ============================================
# STEP 9: Compare both models
# ============================================
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE ($)': [lr_mae, rf_mae],
    'RMSE ($)': [lr_rmse, rf_rmse],
    'R²': [lr_r2, rf_r2]
})

print('=== MODEL COMPARISON ===')
print(comparison.to_string(index=False))
print()

best = 'Linear Regression' if lr_mae < rf_mae else 'Random Forest'
print(f'🏆 Winner: {best} (lowest MAE)')

## 10. Results Visualization

Let's create professional charts to visualize model performance.

In [ ]:
# ============================================
# STEP 10A: Actual vs Predicted prices
# ============================================
best_preds = lr_preds if lr_mae < rf_mae else rf_preds
best_name = 'Linear Regression' if lr_mae < rf_mae else 'Random Forest'

fig = go.Figure()
fig.add_trace(go.Scatter(x=X_test.index, y=y_test, mode='lines',
                         name='Actual', line=dict(color='#3b82f6', width=2)))
fig.add_trace(go.Scatter(x=X_test.index, y=best_preds, mode='lines',
                         name='Predicted', line=dict(color='#f472b6', width=2, dash='dot')))
fig.update_layout(title=f'{ticker} — Actual vs Predicted ({best_name})',
                  xaxis_title='Date', yaxis_title='Price ($)',
                  template='plotly_dark', height=500)
fig.show()

In [ ]:
# ============================================
# STEP 10B: Scatter plot — Actual vs Predicted
# ============================================
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test, y=best_preds, mode='markers',
                         marker=dict(color='#a78bfa', size=8, opacity=0.6),
                         name='Predictions'))
mn = min(y_test.min(), best_preds.min())
mx = max(y_test.max(), best_preds.max())
fig.add_trace(go.Scatter(x=[mn,mx], y=[mn,mx], mode='lines',
                         name='Perfect Prediction', line=dict(color='#22c55e', dash='dash')))
fig.update_layout(title='Prediction Accuracy — Scatter Plot',
                  xaxis_title='Actual Price ($)', yaxis_title='Predicted Price ($)',
                  template='plotly_dark', height=500)
fig.show()

In [ ]:
# ============================================
# STEP 10C: Feature Importance (Random Forest)
# ============================================
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

top = importances.head(12)
fig = go.Figure(go.Bar(
    x=top['importance'].values[::-1],
    y=top['feature'].values[::-1],
    orientation='h', marker_color='#a78bfa'
))
fig.update_layout(title='Top 12 Feature Importances (Random Forest)',
                  xaxis_title='Importance', template='plotly_dark', height=500)
fig.show()

In [ ]:
# ============================================
# STEP 10D: Residual analysis
# ============================================
residuals = y_test.values - best_preds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual distribution
axes[0].hist(residuals, bins=30, color='#a78bfa', edgecolor='white', alpha=0.8)
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Error ($)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(0, color='red', linestyle='--')

# Residuals over time
axes[1].scatter(range(len(residuals)), residuals, alpha=0.5, color='#3b82f6', s=15)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals Over Time')
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Error ($)')

plt.tight_layout()
plt.show()

print(f'Mean residual: ${np.mean(residuals):.2f} (should be ~0)')
print(f'Std residual:  ${np.std(residuals):.2f}')

## 11. Save Best Model

In [ ]:
# ============================================
# STEP 11: Save model for deployment
# ============================================
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(lr_model, '../models/best_model_lr.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print('Model and scaler saved to models/ folder!')

## 12. Conclusion

### Key Findings

1. **Linear Regression outperformed Random Forest** on this time-series task
   - LR can extrapolate linear trends; RF tree splits cannot handle price drift
   - This is a common finding with time series containing lag features

2. **Close_lag_1 (yesterday's close) is the strongest predictor**
   - Stock prices are highly autocorrelated — tomorrow's price is very close to today's
   - This is why the model achieves high accuracy

3. **Feature engineering is critical**
   - Raw prices alone would cause data leakage
   - Technical indicators (SMA, RSI) capture momentum and trend signals

4. **Chronological splitting is essential**
   - Random splits would leak future data into training
   - Always split time series data in order!

### What I Learned
- How to fetch real financial data using APIs
- The importance of preventing data leakage in ML
- Feature engineering with financial domain knowledge
- Model evaluation beyond just accuracy (MAE, RMSE, R²)
- Building interactive dashboards with Streamlit

---
**Task 2 — DevelopersHub Corporation AI/ML Engineering Internship**

Built by Ammar Akbar | May 2026